# **AI Story Generator using GRU + TinyStories Dataset**

## **Project Overview**

**This project builds an AI Story Generator using:**

* GRU (Gated Recurrent Unit)
* TensorFlow/Keras
* TinyStories Dataset from Hugging Face

The model learns story patterns and generates new stories word-by-word.

In [ ]:
!pip install datasets tensorflow -q

## **Import Libraries**

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from datasets import load_dataset

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import os
import random

## **Set Random Seed**

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random Seed Set:", SEED)

Random Seed Set: 42


In [ ]:
dataset = load_dataset("roneneldan/TinyStories")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


## **Extract Stories**

In [ ]:
stories = []

# Take first 1000 stories
for item in dataset['train'].select(range(1000)):
    stories.append(item['text'])

print("Total Stories:", len(stories))
print("\nSample Story:\n")
print(stories[0][:1000])

Total Stories: 1000

Sample Story:

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


## **Combine Text**

In [ ]:
text_data = " ".join(stories)

print("Total Characters:", len(text_data))

Total Characters: 942639


## **Tokenization**

In [ ]:
vocab_size = 5000

# Create tokenizer

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts([text_data])

# Convert text to sequence
sequence_data = tokenizer.texts_to_sequences([text_data])[0]

print("Total Tokens:", len(sequence_data))

Total Tokens: 183978


## **Create Input Sequences**

In [ ]:
sequence_length = 20

input_sequences = []

for i in range(sequence_length, len(sequence_data)):
    seq = sequence_data[i-sequence_length:i+1]
    input_sequences.append(seq)

input_sequences = np.array(input_sequences)

print("Shape:", input_sequences.shape)

Shape: (183958, 21)


## **Split X and y**

In [ ]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (183958, 20)
y Shape: (183958,)


## **Build GRU Model**

In [ ]:
model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_shape=(sequence_length,)
    )
)

# First GRU
model.add(GRU(128, return_sequences=True))
model.add(Dropout(0.2))

# Second GRU
model.add(GRU(128))

# Output Layer
model.add(Dense(vocab_size, activation='softmax'))

# Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Show summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 128)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 20, 128)        │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5000)           │       645,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,483,144 (5.66 MB)

 Trainable params: 1,483,144 (5.66 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X,
    y,
    epochs=30,
    batch_size=32
)

Epoch 1/30
5749/5749 ━━━━━━━━━━━━━━━━━━━━ 456s 79ms/step - accuracy: 0.1432 - loss: 5.2721
Epoch 2/30
5749/5749 ━━━━━━━━━━━━━━━━━━━━ 451s 78ms/step - accuracy: 0.2275 - loss: 4.2842
Epoch 3/30
5749/5749 ━━━━━━━━━━━━━━━━━━━━ 451s 78ms/step - accuracy: 0.2562 - loss: 3.9443
Epoch 4/30
5749/5749 ━━━━━━━━━━━━━━━━━━━━ 450s 78ms/step - accuracy: 0.2722 - loss: 3.7360
Epoch 5/30
5749/5749 ━━━━━━━━━━━━━━━━━━━━ 501s 78ms/step - accuracy: 0.2838 - loss: 3.5808
Epoch 6/30
5749/5749 ━━━━━━━━━━━━━━━━━━━━ 449s 78ms/step - accuracy: 0.2935 - loss: 3.4565
Epoch 7/30
5749/5749 ━━━━━━━━━━━━━━━━━━━━ 449s 78ms/step - accuracy: 0.3018 - loss: 3.3534
Epoch 8/30
5749/5749 ━━━━━━━━━━━━━━━━━━━━ 446s 78ms/step - accuracy: 0.3108 - loss: 3.2667
Epoch 9/30
5749/5749 ━━━━━━━━━━━━━━━━━━━━ 450s 78ms/step - accuracy: 0.3177 - loss: 3.1927
Epoch 10/30
5749/5749 ━━━━━━━━━━━━━━━━━━━━ 450s 78ms/step - accuracy: 0.3242 - loss: 3.1273
Epoch 11/30
5749/5749 ━━━━━━━━━━━━━━━━━━━━ 450s 78ms/step - accuracy: 0.3308 - loss: 3.06